In [71]:
csv_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.csv"
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cbd"
processed = "home/theo/PycharmProjects/Bookoflife/Original/data/processed"

In [2]:
import re
import pandas as pd
from pathlib import Path

def parse_cdb(cdb_path):
    """Parse NLSY79 codebook (.cdb) - simple version"""
    codebook = {}
    text = Path(cdb_path).read_text()

    # Split by long dashes
    blocks = re.split(r"-{40,}", text)

    for block in blocks:
        # Match header: H00016.00    [H40-BPAR-1]    Survey Year: XRND
        m = re.match(r"\s*(\w+\.\d+)\s+\[([^\]]+)\]\s+Survey Year:\s*(\S+)", block)
        if not m:
            continue

        rnum = m.group(1).replace(".", "")  # H0001600
        qname = m.group(2)                   # H40-BPAR-1
        year = m.group(3)                    # XRND

        # Extract question (after "PRIMARY VARIABLE")
        question = ""
        qm = re.search(r"PRIMARY VARIABLE\s*\n\s*\n\s*(.+?)(?:\n\s*\n|$)", block, re.DOTALL)
        if qm:
            question = qm.group(1).strip().replace("\n", " ")

        codebook[rnum] = {
            "rnum": rnum,
            "qname": qname,
            "year": year,
            "question": question,
        }

    return codebook


In [3]:
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cdb"
codebook = parse_cdb(cdb_path)

cb_df = pd.DataFrame(codebook.values())


In [4]:
invalid_year = ~cb_df["year"].astype(str).str.fullmatch(r"\d{4}")

unique_qname = cb_df.groupby("qname")["qname"].transform("size") == 1

# retroperspecitv questions
living_struc = (
    cb_df["year"].astype(str).eq("1988")
    & cb_df["question"]
        .astype(str)
        .str.lower()
        .str.startswith(("lived with", "lived in"), na=False)
)

##sind die ganzen 1988 struktur variablen auch drinnen -> wie identifizieren wir die? questinon text ist mit lived with..
cbd_p = cb_df[unique_qname & ~living_struc & ~(cb_df["year"] == "XRND")] #XRND variablen raus da kein informationsverlust
cbd_pl = cb_df[~(invalid_year | unique_qname) | (cb_df["rnum"] == "R0000100")]
cbd_ls = cb_df[living_struc | (cb_df["rnum"] == "R0000100")]

In [36]:
print(len(cbd_p))
print(len(cbd_pl))
print(len(cbd_ls))


25
181
255


In [37]:
df_csv = pd.read_csv(csv_path)
csv_pl = df_csv[cbd_pl["rnum"]]
csv_p = df_csv[cbd_p["rnum"]]
csv_ls = df_csv[cbd_ls["rnum"]]

In [42]:
csv_p

,R0000100,R0000600,R0002000,R0002300,R0006100,R0006400,R0006500,R0008800,R0173600,R0214700,...,R2738100,R5537600,R7455300,R7456000,R7456100,R7456400,R7457100,R7457200,R8164700,R8165400
0,1,20,-4,-4,2,-4,8,-4,5,3,...,-5,-5,-5,-5,-5,-5,-5,-5,-5,-5
1,2,20,-4,-4,2,-4,5,-4,5,3,...,-4,-4,-4,-4,-4,-4,-4,-4,-4,-4
2,3,17,-4,-4,1,-4,10,-4,5,3,...,0,-4,-4,-4,-4,-4,-4,-4,-5,-5
3,4,16,-4,-4,1,-4,11,-4,5,3,...,-4,-4,-5,-5,-5,-5,-5,-5,-5,-5
4,5,19,-4,-4,1,-4,12,-4,1,3,...,-4,-5,-5,-5,-5,-5,-5,-5,-5,-5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12681,12682,19,-4,-4,1,-4,12,0,15,3,...,-5,-5,-5,-5,-5,-5,-5,-5,-5,-5
12682,12683,19,-4,-4,1,-4,12,0,15,3,...,-5,-5,-5,-5,-5,-5,-5,-5,-5,-5
12683,12684,19,-4,-4,1,-4,9,1,15,3,...,-5,-5,-5,-5,-5,-5,-5,-5,-5,-5
12684,12685,22,-4,-4,1,-4,8,-4,16,2,...,-5,-5,-5,-5,-5,-5,-5,-5,-5,-5


In [62]:
#variables we want to make to long
rnum_vars_p = cbd_p["rnum"].tolist()
# melt: von wide zu long. id_vars ist  'R0000100'
melted_p = csv_p.melt(id_vars=["R0000100"], value_vars=rnum_vars_p,
                     var_name="rnum", value_name="value")


melted_p = melted_p.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")

melted_p = melted_p.rename(columns={"R0000100": "caseid"})
melted_p = melted_p.sort_values(["caseid", "rnum"]).reset_index(drop=True)
#ersten von 78SCRN durch 1979 damit man jahre als str verwenden kann
melted_p["year"] = melted_p["year"].astype(str).str.replace("78SCRN", "1979")
melted_p["syear"] = pd.to_numeric(melted_p["year"], errors="coerce")

In [48]:
pivoted_p = melted_p.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()

In [49]:
pivoted_p["birthyear"] =  1979 - pivoted_p["FAM-1B"]


In [11]:
rnum_vars = cbd_pl["rnum"].tolist()
print(len(rnum_vars))
melted = csv_pl.melt(id_vars=["R0000100"], value_vars=rnum_vars,
                     var_name="rnum", value_name="value")

melted = melted.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")


melted = melted.rename(columns={"R0000100": "caseid"})
melted = melted.sort_values(["caseid", "rnum"]).reset_index(drop=True)


melted["syear"] = pd.to_numeric(melted["year"], errors="coerce")


181


In [12]:
pivoted_pl = melted.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()


In [13]:
rnum_vars_ls = cbd_ls["rnum"].tolist()

# melt: von wide zu long. id_vars ist  'R0000100'
melted_ls = csv_ls.melt(id_vars=["R0000100"], value_vars=rnum_vars_ls,
                     var_name="rnum", value_name="value")

melted_ls = melted_ls.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")

melted_ls = melted_ls.rename(columns={"R0000100": "caseid"})
melted_ls = melted_ls.sort_values(["caseid", "rnum"]).reset_index(drop=True)

melted_ls["syear"] = pd.to_numeric(melted_ls["year"], errors="coerce")

In [14]:
pivoted_ls = melted_ls.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()

In [15]:
def reshape_retrospective(df, var_prefix):
    df = df.dropna(subset=['birthyear'])

    cols = []
    for col in df.columns:
        if col.startswith(var_prefix.upper()):
            # Extrahiere die Zahl am Ende
            match = re.search(r'(\d+)$', col)
            if match:
                age_num = int(match.group(1))
                if 0 <= age_num <= 18:
                    cols.append(col)
    if not cols:
        print(f"Keine passenden Spalten gefunden!")
        return pd.DataFrame()

    long = df.melt(
        id_vars=['caseid', 'year', 'birthyear'],
        value_vars=cols,
        var_name='suffix',
        value_name=var_prefix
    )

    long['age'] = long['suffix'].str.extract(r'(\d+)$').astype(int)
    long['year'] = long['birthyear'] + long['age']  # ✓ direkt als 'year'

    return long[['caseid', 'year', 'age', 'birthyear', var_prefix]].sort_values(['caseid', 'year']).reset_index(drop=True)

In [16]:
# Merge mit caseid  year
pivoted_ls_with_birth = pivoted_ls.merge(
    pivoted_p[['caseid', 'birthyear']],  # Nur eindeutige caseid-birthyear
    on='caseid',
    how='left'
)
# Anwenden
adopt_long = reshape_retrospective(pivoted_ls_with_birth, 'adoptd')


In [17]:
cols_ending_18 = [col for col in pivoted_ls_with_birth.columns
                  if re.search(r'18$', col)]
prefixes = [re.sub(r'\d+$', '', col) for col in cols_ending_18]

long_dfs = [reshape_retrospective(pivoted_ls_with_birth, prefix) for prefix in prefixes]

result_ls = pd.concat(long_dfs, axis=1)
result_ls = result_ls.loc[:, ~result_ls.columns.duplicated()].sort_values(['caseid', 'year']).reset_index(drop=True)


In [53]:
result_ls['year'] = result_ls['year'].astype(int)
pivoted_pl['year'] = pivoted_pl['year'].astype(int)
pivoted_p['year'] = pivoted_p['year'].astype(int)

merged_1 = result_ls.merge(pivoted_pl, on=['caseid', 'year'], how='outer')

merged_1['birthyear'] = merged_1.groupby('caseid')['birthyear'].transform(
    lambda x: x.fillna(x.dropna().iloc[0] if len(x.dropna()) > 0 else None))
merged_1['age'] = pd.to_numeric(merged_1['year'], errors='coerce') - merged_1['birthyear']



In [54]:
merged_1

,caseid,year,age,birthyear,ADOPTD,ADOPTM,BDAD,BMOM,CHLDHM,DTENT,...,Q9-91.03,Q9-91.04,Q9-91.05,Q9-91.06,Q9-91.09,Q9-91_1,Q9-91_2,Q9-92.01,Q9-92_1,Q9-92_2
0,1,1959,0.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1960,1.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1961,2.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1962,3.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1963,4.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507323,12686,2010,49.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,-5.0,NaN,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507324,12686,2012,51.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507325,12686,2014,53.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,-5.0,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507326,12686,2016,55.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
# Merge mit pivoted_p
final_result = merged_1.merge(pivoted_p, on=['caseid', 'year'], how='outer')
final_result = final_result.sort_values(['caseid', 'year']).reset_index(drop=True)
final_result = final_result.rename(columns={"birthyear_x": "birthyear"})
final_result = final_result.drop(columns=['birthyear_y'])

final_result['birthyear'] = final_result.groupby('caseid')['birthyear'].transform(
    lambda x: x.fillna(x.dropna().iloc[0] if len(x.dropna()) > 0 else None))

final_result["age"] = final_result["year"] - final_result["birthyear"]


In [58]:
final_result

,caseid,year,age,birthyear,ADOPTD,ADOPTM,BDAD,BMOM,CHLDHM,DTENT,...,Q9-89.08,Q9-89.10,Q9-90.04,Q9-91.07,Q9-91.08,Q9-91.10,Q9-92.02,SAMPLE_ID,SAMPLE_RACE,SAMPLE_SEX
0,1,1959,0.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1960,1.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1961,2.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1962,3.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1963,4.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519059,12686,2010,49.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
519060,12686,2012,51.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
519061,12686,2014,53.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
519062,12686,2016,55.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:

# Erstelle eine "schlanke" Version: Vergleiche mit der Zeile darüber (nicht Baseline)
def to_sparse_format(df):
    """Nur Änderungen zwischen aufeinanderfolgenden Zeilen"""
    result = []

    for caseid in df['caseid'].unique():
        person_df = df[df['caseid'] == caseid].sort_values('year').reset_index(drop=True)

        for i, row in person_df.iterrows():
            sparse_row = {'caseid': caseid, 'year': row['year']}

            # Erste Zeile: immer alle Werte nehmen
            if i == 0:
                for col in person_df.columns:
                    if col not in ['caseid', 'year']:
                        sparse_row[col] = row[col]
            else:
                # Vergleiche mit vorheriger Zeile
                prev_row = person_df.iloc[i-1]
                for col in person_df.columns:
                    if col not in ['caseid', 'year']:
                        # Nimm Wert wenn: unterschiedlich von vorher ODER neu vorhanden (war NaN, ist jetzt nicht)
                        if pd.isna(prev_row[col]) or pd.isna(row[col]) or prev_row[col] != row[col]:
                            sparse_row[col] = row[col]

            result.append(sparse_row)

    return pd.DataFrame(result)

merged_1_sparse = to_sparse_format(merged_1)
merged_1_sparse

,caseid,year,age,birthyear,ADOPTD,ADOPTM,BDAD,BMOM,CHLDHM,DTENT,...,Q9-91.03,Q9-91.04,Q9-91.05,Q9-91.06,Q9-91.09,Q9-91_1,Q9-91_2,Q9-92.01,Q9-92_1,Q9-92_2
0,1,1959.0,0.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1960.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1961.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1962.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1963.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507323,12686,2010.0,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507324,12686,2012.0,51.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507325,12686,2014.0,53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-5.0,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507326,12686,2016.0,55.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
%cd processed

final_result.to_csv("nlsy79_reshaped_pl_p.csv", index=False)
merged_1.to_csv("nlsy79_pl.csv", index=False)
pivoted_p.to_csv("nlsy79_p.csv", index=False)
merged_1_sparse.to_csv("nlsy79_pl_sparse.csv", index=False)

[Errno 2] No such file or directory: 'processed'
/home/theo/PycharmProjects/Bookoflife/Original/data/processed
